# Hyponatremia Correction Algorithm — Formula Derivation

## Background

Dysnatremia (abnormal serum sodium) is one of the most common electrolyte disorders in hospitalized patients. Correcting serum sodium too slowly risks ongoing neurological injury from cerebral edema, while correcting too rapidly risks **osmotic demyelination syndrome (ODS)** — a devastating and often irreversible neurological complication.

Clinicians need a reliable way to answer two questions:
1. **What IVF rate is needed** to reach a target serum sodium over a given time?
2. **What serum sodium will result** from a planned infusion over a given time?

### Limitation of the Adrogué-Madias Equation

The widely used Adrogué-Madias equation estimates the change in serum sodium per liter of infusate:

$$\Delta[\text{Na}^+]_s = \frac{[\text{Na}^+]_{\text{infusate}} - [\text{Na}^+]_{\text{serum}}}{\text{TBW} + 1}$$

This equation treats the body as a **closed system** — it does not account for ongoing urine output or the electrolytes lost in urine. In conditions such as **SIADH**, where patients excrete concentrated urine rich in Na⁺ and K⁺, ignoring urinary losses can substantially underestimate the rate of sodium correction, raising the risk of overcorrection.

### This Approach: An Open System Model

The equations derived here treat the body as an **open system**, explicitly incorporating:
- Urine output rate (`URate`)
- Urine sodium concentration (`UNa`)
- Urine potassium concentration (`UK`)

Urinary potassium is included because K⁺ is the dominant intracellular cation; its renal excretion is osmotically equivalent to Na⁺ retention and therefore raises serum sodium just as urine Na⁺ loss does.

## Variables

| Symbol | Description | Units |
|--------|-------------|-------|
| `NaStart` | Starting serum sodium | mEq/L |
| `NaEnd` | Target (ending) serum sodium | mEq/L |
| `TBW` | Total body water at baseline | L |
| `TES` | Total exchangeable sodium at baseline | mEq |
| `IVFConc` | Sodium concentration of the IVF | mEq/L |
| `IVFRate` | IVF infusion rate | L/hr |
| `UNa` | Urine sodium concentration | mEq/L |
| `UK` | Urine potassium concentration | mEq/L |
| `URate` | Urine output rate | L/hr |
| `Time` | Duration of infusion | hr |

**TBW estimation** (Adrogué-Madias convention):
- Young males: TBW = 0.60 × weight (kg)
- Young females / elderly males: TBW = 0.50 × weight (kg)
- Elderly females: TBW = 0.45 × weight (kg)

## Derivation

### Step 1 — Express total exchangeable sodium (TES) at baseline

Serum sodium is total exchangeable sodium divided by total body water:

$$NaStart = \frac{TES}{TBW} \implies TES = NaStart \times TBW$$

### Step 2 — Track how TES changes over time `T`

Two processes alter TES during the infusion:

| Process | Volume change | Sodium change |
|---------|--------------|---------------|
| IVF infused | +IVFRate × T | +IVFConc × IVFRate × T |
| Urine excreted | −URate × T | −(UNa + UK) × URate × T |

The effective urinary sodium loss includes both Na⁺ and K⁺, because K⁺ excretion in urine is osmotically equivalent to Na⁺ retention in serum.

Therefore at time T:

$$TES_{new} = TES - (UNa + UK) \times URate \times T + IVFConc \times IVFRate \times T$$

$$TBW_{new} = TBW - URate \times T + IVFRate \times T$$

### Step 3 — Write the expression for NaEnd

$$NaEnd = \frac{TES_{new}}{TBW_{new}} = \frac{TES - (UNa + UK) \times URate \times T + IVFConc \times IVFRate \times T}{TBW - URate \times T + IVFRate \times T}$$

Substituting TES = NaStart × TBW:

$$NaEnd = \frac{NaStart \times TBW - (UNa + UK) \times URate \times T + IVFConc \times IVFRate \times T}{TBW - URate \times T + IVFRate \times T}$$

### Step 4 — Solve for the two clinical questions

The system of two equations (Step 1 and Step 3) can be solved for either:
- **IVFRate**, given a desired NaEnd → *how fast should I infuse?*
- **NaEnd**, given a planned IVFRate → *what sodium will result?*

SymPy is used below to perform this algebra exactly.

In [ ]:
from sympy import *

# Symbols
NaStart, NaEnd, UNa, UK, URate = symbols('NaStart NaEnd UNa UK URate', real=True)
TES, TBW = symbols('TES TBW', positive=True)
IVFConc, IVFRate, Time = symbols('IVFConc IVFRate Time', positive=True)

# Equation 1: baseline serum sodium (TES definition)
eq1 = Eq(NaStart, TES / TBW)

# Equation 2: serum sodium after infusion (open system — urine losses included)
eq2 = Eq(
    NaEnd,
    (TES - (UNa + UK) * URate * Time + IVFConc * IVFRate * Time)
    / (TBW - URate * Time + IVFRate * Time)
)

print("Equation 1:", eq1)
print("Equation 2:", eq2)

## Solution 1 — IVF Rate Required to Reach Target Sodium

Solve the system for `IVFRate`, eliminating `TES`.

In [ ]:
solution_rate = solve([eq1, eq2], [IVFRate, TES])
ivf_rate_expr = solution_rate[IVFRate]

print("IVFRate =")
pprint(ivf_rate_expr)
print()
print("LaTeX:")
print(latex(ivf_rate_expr))

## Solution 2 — Predicted Serum Sodium After a Planned Infusion

Solve the system for `NaEnd`, eliminating `TES`.

In [ ]:
solution_na = solve([eq1, eq2], [NaEnd, TES])
na_end_expr = solution_na[NaEnd]

print("NaEnd =")
pprint(na_end_expr)
print()
print("LaTeX:")
print(latex(na_end_expr))

## Final Formulas

### IVF Rate to Achieve a Target Serum Sodium

$$IVFRate = \frac{NaEnd \times TBW - NaEnd \times Time \times URate - NaStart \times TBW + Time \times (UK + UNa) \times URate}{Time \times (IVFConc - NaEnd)}$$

**When to use:** You know the desired correction target (`NaEnd`) and want to calculate the infusion rate.

---

### Predicted Serum Sodium After a Planned Infusion

$$NaEnd = \frac{IVFConc \times IVFRate \times Time + NaStart \times TBW - Time \times (UK + UNa) \times URate}{IVFRate \times Time + TBW - Time \times URate}$$

**When to use:** You know the planned infusion rate (`IVFRate`) and want to predict the resulting sodium.

---

## Clinical Notes

- **Setting URate = 0** (anuric patient) reduces both formulas to a closed-system model equivalent to Adrogué-Madias.
- **Setting UNa = UK = 0** with non-zero URate models infusion of free water only (e.g., primary polydipsia with electrolyte-free urine).
- In SIADH, urine (UNa + UK) often exceeds IVFConc; this drives sodium up faster than the infusate concentration alone would predict, explaining why the Adrogué-Madias equation underestimates correction speed in this setting.
- **These formulas are planning tools.** Serum sodium must be monitored frequently. Correction should generally not exceed 10 mEq/L per 24 hours (8 mEq/L in high-risk patients).